# $b_e$ sweep — spiking + mean-field + whole-brain dynamics (Sacha et al. 2025, Fig 4 style)

Reproduces the methodology of Sacha *et al.*, *Nat. Comp. Sci.* (2025) Figure 4 across the AdEx
spike-triggered adaptation parameter $b_e$, using **the canonical toolkit functions only** — no
parallel re-implementations. All three model scales share one OU-correlated external drive
realization (same noise, per-scale baseline per the Fig 4a caption).

## Stack

| Scale                   | Function                                                                                              | Engine                |
|-------------------------|-------------------------------------------------------------------------------------------------------|-----------------------|
| Spiking (SNN)           | [`tvbtoolkit.brian_mf.adex.network.run_adex_network_simulation`](../src/tvbtoolkit/brian_mf/adex/network.py) | Brian2                |
| Single-region mean field| [`tvbtoolkit.whole_brain.simulation.run_whole_brain_simulation`](../src/tvbtoolkit/whole_brain/simulation.py) on a 1×1 zero-coupling connectivity, `zerlaut_order=2` | TVB Zerlaut 2nd-order |
| Whole brain (DK-68)     | Same as above with the packaged `data/connectivity/connectivity_68.zip` atlas, `zerlaut_order=2`       | TVB Zerlaut 2nd-order |

Both TVB runs route through the same `_Zerlaut2OUDrive` subclass (defined in
`scripts/b_sweep_publication_run.py`) which overrides `dfun` to set
`external_input_ex_ex` / `external_input_in_ex` from the shared OU trace at
every integrator call. TVB internal stochastic noise is disabled
(`weight_noise = 0`, `stochastic_integrator = False` / HeunDeterministic) so
the shared OU trace is the **only** stochastic source — same as the SNN.

## Parameters (lab v2, `BASE_PARAMETER_MODEL_NEW`)

Shared, for each $b_e$:

* `N_tot = 10000`, `g = 0.2` (20% inhibitory), `p_connect = 0.05`
* `E_L_e = -63 mV`, `E_L_i = -65 mV` *(lab v2 refinement; paper uses -64/-65)*
* `tau_w_e = 500 ms`, `Cm = 200 pF`, `Gl = 10 nS`, `Q_e = 1.5 nS`, `Q_i = 5.0 nS`
* `T = 20 ms` mean-field timescale; v2 `P_e` / `P_i` polynomial coefficients
* MF/WB initial conditions: `E = 0.004 KHz` (4 Hz), `I = 0.010 KHz`, `W_e = 50 pA`
* WB coupling: Linear, `a = 0.3`

## $b_e$ sweep range (paper Fig 4a reference)

Fig 4a uses $b_e=5$ pA (AI/awake state) and $b_e=120$ pA (UD/anesthesia state).
This notebook sweeps **7 values from AI to deep UD**:

$$b_e \in \{5,\ 25,\ 45,\ 65,\ 85,\ 105,\ 125\}\ \text{pA}$$

evenly spaced at 20 pA. Simulations are 22 s long; the first 4 s are discarded as transient.

## External-drive convention (paper Fig 4a)

Sacha *et al.* 2025 Eq. (20):

$$v_\text{aff}(t) = v_\text{drive} + \sigma \cdot \xi(t),\quad d\xi = -\xi/\tau_\text{OU}\,dt + dW$$

with paper $\sigma=3.5$ Hz, $\tau_\text{OU}=5$ ms.

**Per-scale $v_\text{drive}$ baselines** from the Fig 4a caption:

| Model | $v_\text{drive}$ (Hz) | Afferents per post-neuron | Effective baseline Hz/neuron |
|-------|-----------------------|---------------------------|------------------------------|
| SNN + single-region MF | **0.4**   | 400 | 160 |
| Whole brain (DK-68)    | **0.315** | 400 | 126 |

Both traces use the **same OU noise realization** (same seed → same $\xi(t)$);
only the baseline differs. The SNN gets its drive through `external_rate_hz_trace=`;
the MF and WB get theirs through `external_input_ex_ex` / `external_input_in_ex`
set per dfun call by `_Zerlaut2OUDrive`.

## Run

All heavy lifting lives in two scripts, runnable independently from the CLI:

* `scripts/b_sweep_publication_run.py` — runs SNN + MF + WB for each $b_e$, pickles per-b results
* `scripts/b_sweep_publication_figure.py` — builds the 4 × 7 publication figure from pickles

```bash
# Full sweep (fresh, ~1 h on a workstation):
python scripts/b_sweep_publication_run.py --all --force

# Single b for iteration:
python scripts/b_sweep_publication_run.py --b 25 --force

# Re-run only one block — keeps the others:
python scripts/b_sweep_publication_run.py --all --mf-only
python scripts/b_sweep_publication_run.py --all --wb-only
python scripts/b_sweep_publication_run.py --all --snn-only

# Build the figure:
python scripts/b_sweep_publication_figure.py
```

The cells below offer the same as Python calls.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from b_sweep_publication_run import run_one_b, B_VALUES_PA
from b_sweep_publication_figure import build_figure

print("b_e values to sweep:", B_VALUES_PA)

## Run the sweep

Each `run_one_b` call writes a pickle to `notebooks/outputs/b_sweep/per_b/b_<b>.pkl` and skips
if it already exists. Pass `force=True` to overwrite.

In [ ]:
for b in B_VALUES_PA:
    run_one_b(b)

## Build the figure

In [ ]:
png_path = build_figure()

from IPython.display import Image, display
display(Image(filename=str(png_path)))